# Search tool with Gemini 2.x

In this tutorial you are going to leverage the latest search tool of the Gemini 2.x model to write a company report. 

You may be asking, why does one need to use the search tool for this purpose? Well, as you may be aware, today's business world evolves very fast and LLMs generally are not trained frequently enough to capture the latest updates. Luckily Google search comes to the rescue. Google search is built to provide accurate and nearly realtime information and can help us fulfill this task perfectly.


## Setup

### Install the Google GenAI SDK

Install the Google GenAI SDK from [npm](https://www.npmjs.com/package/@google/genai). 

```bash
$ npm install @google/genai
```

### Setup your API key

You can [create](https://aistudio.google.com/app/apikey) your API key using Google AI Studio with a single click.

Remember to treat your API key like a password. Don't accidentally save it in a notebook or source file you later commit to GitHub. In this notebook we will be storing the API key in a `.env` file. You can also set it as an environment variable or use a secret manager. 

Here's how to set it up in a `.env` file:

```bash
$ touch .env
$ echo "GEMINI_API_KEY=<YOUR_API_KEY>" >> .env
```

:::{.callout-tip}

Another option is to set the API key as an environment variable. You can do this in your terminal with the following command:

```bash
$ export GEMINI_API_KEY="<YOUR_API_KEY>"
```
:::

### Load the API key

To load the API key from the `.env` file, we will use the `dotenv` package. This package loads environment variables from a `.env` file into `process.env`. 

```bash
$ npm install dotenv
```

Then, we can load the API key in our code:


In [2]:
const dotenv = require("dotenv") as typeof import("dotenv");

dotenv.config({
  path: "../.env",
});

const GEMINI_API_KEY = process.env.GEMINI_API_KEY ?? "";
if (!GEMINI_API_KEY) {
  throw new Error("GEMINI_API_KEY is not set in the environment variables");
}
console.log("GEMINI_API_KEY is set in the environment variables");


GEMINI_API_KEY is set in the environment variables


:::{.callout-note}
In our particular case the `.env` is is one directory up from the notebook, hence we need to use `../` to go up one directory. If the `.env` file is in the same directory as the notebook, you can omit it altogether. 

```
│
├── .env
└── examples
    └── Search_grounding_for_research_report.ipynb
```
:::


### Initialize SDK Client

With the new SDK, now you only need to initialize a client with you API key (or OAuth if using [Vertex AI](https://cloud.google.com/vertex-ai)). The model is now set in each call.


In [3]:
const google = require("@google/genai") as typeof import("@google/genai");

const ai = new google.GoogleGenAI({ apiKey: GEMINI_API_KEY });


### Select a model

Now select the model you want to use in this guide, either by selecting one in the list or writing it down. Keep in mind that some models, like the 2.5 ones are thinking models and thus take slightly more time to respond (cf. [thinking notebook](../quickstarts/Get_started_thinking.ipynb) for more details and in particular learn how to switch the thiking off).


In [4]:
const tslab = require("tslab") as typeof import("tslab");

const MODEL_ID = "gemini-2.5-flash-preview-05-20";


## Write the report with Gemini 2.x

### Select a target company to research

Next you will use Alphabet as an example research target.


In [5]:
const COMPANY = "Alphabet Inc.";


### Have Gemini 2.x plan for the task and write the report

Gemini 2.0 is a huge step up from previous models since it can reason, plan, search and write in one go. In this case, you will only give Gemini 2.0 a prompt to do all of these and the model will finish your task seamlessly. You will also using output streaming, which streams the response as it is being generated, and the model will return chunks of the response as soon as they are generated. If you would like the SDK to return a response after the model completes the entire generation process, you can use the non-streaming approach as well.

Note that you must enable Google Search Suggestions, which help users find search results corresponding to a grounded response, and [display the search queries](https://ai.google.dev/gemini-api/docs/grounding/search-suggestions#display-requirements) that are included in the grounded response's metadata. You can find [more details](https://ai.google.dev/gemini-api/terms#grounding-with-google-search) about this requirement.


In [9]:
const SYSTEM_PROMPT = `
  You are an analyst that conducts company research.
  You are given a company name, and you will work on a company report. You have access
  to Google Search to look up company news, updates and metrics to write research reports.
  
  When given a company name, identify key aspects to research, look up that information
  and then write a concise company report.
  
  Feel free to plan your work and talk about it, but when you start writing the report,
  put a line of dashes (---) to demarcate the report itself, and say nothing else after
  the report has finished.
`;

const responseStream = await ai.models.generateContentStream({
  model: MODEL_ID,
  contents: COMPANY,
  config: {
    systemInstruction: SYSTEM_PROMPT,
    temperature: 0,
    tools: [{ googleSearch: {} }],
  },
});

let report = "";
const searchQueries: string[] = [];
for await (const chunk of responseStream) {
  const candidate = chunk.candidates[0];

  for (const part of candidate.content.parts) {
    if (part.text) {
      tslab.display.markdown(part.text);

      // Find and save the report itself.
      const match = /(^|\n)-+\n(.*)$/s.exec(part.text);
      if (match) {
        // Find the starting '---' line and start saving.
        report += match[2];
      } else if (report.length > 0) {
        // If there's already something in the buffer, keep recording.
        report += part.text;
      }
    } else {
      console.log(JSON.stringify(part, null, 2));
    }
  }

  // You must enable Google Search Suggestions
  if (candidate.groundingMetadata?.searchEntryPoint) {
    tslab.display.html(candidate.groundingMetadata.searchEntryPoint.renderedContent ?? "");
    searchQueries.push(candidate.groundingMetadata.searchEntryPoint.renderedContent ?? "");
  }
}


---
## Company Report: Alphabet Inc.

Alphabet Inc. is an American multinational technology conglomerate holding company headquartered in Mountain View, California. Established on October 2, 2

015, through a restructuring of Google, Alphabet became the parent holding company for Google and several of its former subsidiaries. This strategic move aimed to make the core Google business "cleaner and more accountable"

 while granting greater autonomy to other ventures operating outside of internet services. Alphabet is recognized as one of the "Big Five" American information technology companies, alongside Amazon, Apple, Meta, and Microsoft.



The company's shares are traded on the Nasdaq under the ticker symbols GOOGL (Class A, with voting rights) and GOOG (Class C, non-voting). As of July 20

25, Alphabet holds a market capitalization of approximately $2.192 trillion USD, making it the world's 5th most valuable company by market cap. Forbes ranked Alphabet 9th in

...

, which now serves 1.5 billion users monthly. Google Cloud's AI-powered portfolio is also experiencing significant demand. In March 2025, Alphabet agreed to acquire cloud security start-

up Wiz in a $32 billion deal.

However, Alphabet also faces ongoing challenges, including competitive pressures in AI-powered search and increased regulatory scrutiny.

 The company is currently navigating a US Justice Department antitrust case concerning its search monopoly and is under investigation by UK and EU regulators regarding its dominance in mobile ecosystems. In January 2023, Alphabet announced

 layoffs affecting approximately 12,000 jobs, or 6% of its global workforce.

Alphabet Inc. market position 
 Alphabet Inc. recent financial performance 
 Alphabet Inc. overview 
 Alphabet Inc. key products and services 
 Alphabet Inc. leadership 
 Alphabet Inc. recent news and developments

Very impressive! Gemini 2.0 starts by planning for the task, performing relevant searches and streams out a report for you.


## Final output

Render the final output.


In [10]:
tslab.display.markdown("## Final report");
tslab.display.markdown(report);
tslab.display.markdown("## Search queries");
searchQueries.forEach((query) => {
  tslab.display.html(query);
});


## Final report

## Company Report: Alphabet Inc.

Alphabet Inc. is an American multinational technology conglomerate holding company headquartered in Mountain View, California. Established on October 2, 2015, through a restructuring of Google, Alphabet became the parent holding company for Google and several of its former subsidiaries. This strategic move aimed to make the core Google business "cleaner and more accountable" while granting greater autonomy to other ventures operating outside of internet services. Alphabet is recognized as one of the "Big Five" American information technology companies, alongside Amazon, Apple, Meta, and Microsoft.

The company's shares are traded on the Nasdaq under the ticker symbols GOOGL (Class A, with voting rights) and GOOG (Class C, non-voting). As of July 2025, Alphabet holds a market capitalization of approximately $2.192 trillion USD, making it the world's 5th most valuable company by market cap. Forbes ranked Alphabet 9th in its 2025 Global 2000 list, notably placing it first in profits.

### Products and Services

Alphabet's diverse portfolio is primarily organized into three main segments: Google Services, Google Cloud, and Other Bets.

*   **Google Services:** This is the largest segment, accounting for 92% of Alphabet's annual revenue in 2022. It encompasses Google's core search and advertising businesses, along with widely used products and platforms such as Android, Chrome, Google Maps, Google Play, YouTube, Gmail, and Google Nest smart home devices. The segment also includes Google Workspace productivity applications and generative artificial intelligence (AI) offerings like Bard and Gemini.
*   **Google Cloud:** This segment provides a suite of enterprise-level cloud computing solutions for businesses and governments. It contributed 7% of Alphabet's revenue in 2022 and competes with major players like Amazon Web Services (AWS) and Microsoft Azure.
*   **Other Bets:** Representing 1% of 2022 revenue, this segment houses Alphabet's emerging and more speculative businesses. Notable ventures include Waymo (self-driving car technology), Calico (biotechnology research focused on longevity), X (an R&D lab incubating projects like drone delivery service Wing), CapitalG (a private equity fund), DeepMind (AI research), and Verily (life sciences research).

### Financial Performance

Alphabet has demonstrated robust financial growth. For the first quarter of 2025 (ended March 31, 2025), consolidated revenues increased by 12% year-over-year (14% in constant currency) to $90.2 billion. Google Services revenues grew by 10% to $77.3 billion, while Google Cloud revenues saw a significant 28% increase, reaching $12.3 billion. Total operating income rose by 20%, with the operating margin expanding to 34%. Net income increased by 46%, and earnings per share (EPS) grew by 49% to $2.81. The company also announced a 5% increase to its dividend, resulting in a quarterly cash dividend of $0.21.

In the fourth quarter of 2024 (ended December 31, 2024), Alphabet reported consolidated revenues of $96.5 billion, a 12% increase year-over-year. Google Services revenues were $84.1 billion (up 10%), and Google Cloud revenues reached $12.0 billion (up 30%). Operating income increased by 31%, and net income by 28%. Annually, for 2024, Alphabet's revenues were approximately $350 billion, with operating income around $112.4 billion and net income exceeding $100 billion. The company invested approximately $49.3 billion in research and development in 2024.

### Leadership and Recent Developments

Sundar Pichai serves as the Chief Executive Officer of Alphabet Inc., a role he assumed in December 2019, while also retaining his position as CEO of Google. Larry Page and Sergey Brin, the co-founders of Google, stepped down from their executive roles at Alphabet in 2019 but remain active as employees, board members, and controlling shareholders. Other key executives include John L. Hennessy as Chairman, Ruth Porat as President and Chief Investment Officer, and Anat Ashkenazi as Chief Financial Officer.

Recent developments highlight Alphabet's continued focus on AI and its market position. The company surpassed a $2 trillion market valuation for the first time on April 26, 2024, following the announcement of its first-ever dividend payout and a $70 billion stock buyback program. Alphabet is heavily investing in AI, rolling out advanced models like Gemini 2.5 and integrating AI features such as "AI Overviews" into Search, which now serves 1.5 billion users monthly. Google Cloud's AI-powered portfolio is also experiencing significant demand. In March 2025, Alphabet agreed to acquire cloud security start-up Wiz in a $32 billion deal.

However, Alphabet also faces ongoing challenges, including competitive pressures in AI-powered search and increased regulatory scrutiny. The company is currently navigating a US Justice Department antitrust case concerning its search monopoly and is under investigation by UK and EU regulators regarding its dominance in mobile ecosystems. In January 2023, Alphabet announced layoffs affecting approximately 12,000 jobs, or 6% of its global workforce.

## Search queries

Alphabet Inc. market position 
 Alphabet Inc. recent financial performance 
 Alphabet Inc. overview 
 Alphabet Inc. key products and services 
 Alphabet Inc. leadership 
 Alphabet Inc. recent news and developments

As you can see, the Gemini 2.0 model is able to write a concise, accurate and well-structured research report for us. All the information in the report is factual and up-to-date.

Note that this tutorial is meant to showcase the capability of the new search tool and inspire interesting use cases, not to build a production research report generator. If you are looking to use a tool, please check out Google [Deep Research](https://blog.google/products/gemini/google-gemini-deep-research/) in the Gemini app.


## Next Steps

- To learn more about the search tool, check out the [Search tool documentation](https://ai.google.dev/gemini-api/docs/models/gemini-v2#search-tool).
- To get started with streaming, check out the [Streaming Quickstart](../quickstarts/Streaming.ipynb).
- To get started with the search tool, check out the [Search tool quick start](../quickstarts/Search_Grounding.ipynb).
- To see how much easier and more powerful the search tool in Gemini 2.0 is than the previous search grounding feature, compare this tutorial with the [previous example](Search_grounding_for_research_report.ipynb).

To learn more about all the Gemini features, check out the Gemini quickstart notebooks.
